<a href="https://colab.research.google.com/github/fatahrahimi330/XST-Deepfake-Detection/blob/master/Model/BaseLine_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The BaseLine Model Architechture is CNN+ViT
The dataset was loaded from Google Drive and organized into training, validation, and test sets with two classes: fake and real. Each facial image was resized to 224 × 224 pixels and normalized using ImageNet statistics. A hybrid CNN + ViT model was employed, where ResNet-18 extracted local spatial artifacts and ViT-B/16 captured global contextual inconsistencies. The fused feature representation was passed to a fully connected classifier for binary deepfake detection.

## 1. Importing Lobraries

In [ ]:
import os
import copy
import time
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models import vit_b_16, ViT_B_16_Weights

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

## 2. Loading Dataset

### 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_root = "/content/drive/MyDrive/XST-Deepfake-Detection/processed_ffpp"

train_dir = os.path.join(data_root, "train")
val_dir   = os.path.join(data_root, "val")
test_dir  = os.path.join(data_root, "test")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 3. Data Preprocessing

### 1. Transforms

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

### 2. Load datasets

In [ ]:
train_dataset = ImageFolder(train_dir, transform=train_transform)
val_dataset   = ImageFolder(val_dir, transform=val_test_transform)
test_dataset  = ImageFolder(test_dir, transform=val_test_transform)

print("Classes:", train_dataset.classes)
print("Class mapping:", train_dataset.class_to_idx)
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Test size:", len(test_dataset))

Classes: ['fake', 'real']
Class mapping: {'fake': 0, 'real': 1}
Train size: 21645
Val size: 4476
Test size: 5102


### 3. DataLoaders

In [ ]:
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

## 4. Build the Model

In [ ]:
class CNN_ViT(nn.Module):
    def __init__(self, pretrained=True, freeze_backbones=False):
        super(CNN_ViT, self).__init__()

        # CNN branch
        cnn_weights = ResNet18_Weights.DEFAULT if pretrained else None
        self.cnn = resnet18(weights=cnn_weights)
        cnn_feature_dim = self.cnn.fc.in_features   # 512
        self.cnn.fc = nn.Identity()

        # ViT branch
        vit_weights = ViT_B_16_Weights.DEFAULT if pretrained else None
        self.vit = vit_b_16(weights=vit_weights)
        vit_feature_dim = self.vit.heads.head.in_features   # 768
        self.vit.heads = nn.Identity()

        if freeze_backbones:
            for p in self.cnn.parameters():
                p.requires_grad = False
            for p in self.vit.parameters():
                p.requires_grad = False

        fusion_dim = cnn_feature_dim + vit_feature_dim  # 1280

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        cnn_feat = self.cnn(x)              # [B, 512]
        vit_feat = self.vit(x)              # [B, 768]
        fused = torch.cat([cnn_feat, vit_feat], dim=1)   # [B, 1280]
        out = self.classifier(fused)        # [B, 1]
        return out

### Inspect the Model Architecture

In [ ]:
model = CNN_ViT(pretrained=True, freeze_backbones=False).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

In [ ]:
# Loss and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

## 5. Compile and Fit the Model

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    all_labels = []
    all_preds = []

    for images, labels in loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        probs = torch.sigmoid(outputs)
        preds = (probs >= 0.5).float()

        all_labels.extend(labels.cpu().numpy().flatten())
        all_preds.extend(preds.cpu().numpy().flatten())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).float()

            all_labels.extend(labels.cpu().numpy().flatten())
            all_preds.extend(preds.cpu().numpy().flatten())
            all_probs.extend(probs.cpu().numpy().flatten())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_precision = precision_score(all_labels, all_preds, zero_division=0)
    epoch_recall = recall_score(all_labels, all_preds, zero_division=0)
    epoch_f1 = f1_score(all_labels, all_preds, zero_division=0)

    return epoch_loss, epoch_acc, epoch_precision, epoch_recall, epoch_f1, all_labels, all_preds, all_probs

In [ ]:
from tqdm import tqdm
import copy, time, torch

num_epochs = 10
best_val_loss = float("inf")
best_model_wts = copy.deepcopy(model.state_dict())

total_steps = num_epochs * len(train_loader)

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_precision": [],
    "val_recall": [],
    "val_f1": []
}

pbar = tqdm(total=total_steps, desc="Full Training Progress")

for epoch in range(num_epochs):
    start_time = time.time()

    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device).float().unsqueeze(1)   # <-- fix here

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = (torch.sigmoid(outputs) >= 0.5).long()
        correct += (preds == labels.long()).sum().item()
        total += labels.size(0)

        pbar.update(1)

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    val_loss, val_acc, val_precision, val_recall, val_f1, _, _, _ = evaluate(
        model, val_loader, criterion, device
    )

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_precision"].append(val_precision)
    history["val_recall"].append(val_recall)
    history["val_f1"].append(val_f1)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), "/content/best_cnn_vit_ffpp.pth")

    elapsed = time.time() - start_time

    pbar.set_postfix({
        "Epoch": f"{epoch+1}/{num_epochs}",
        "Train Loss": f"{train_loss:.3f}",
        "Val Loss": f"{val_loss:.3f}",
        "Val Acc": f"{val_acc:.3f}",
        "F1": f"{val_f1:.3f}",
        "Time": f"{elapsed:.1f}s"
    })

pbar.close()

Streaming output truncated to the last 5000 lines.
Full Training Progress:  81%|████████  | 5465/6770 [3:29:14<25:10,  1.16s/it, Epoch=8/10, Train Loss=0.089, Val Loss=0.853, Val Acc=0.876, F1=0.897, Time=831.9s]


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

### Check Current GPU Type

In [ ]:
!nvidia-smi

In [ ]:
model.load_state_dict(best_model_wts)
print("Best model loaded.")

## 6. Make Prediction

In [ ]:
def predict_image(model, image_path, transform, device, class_names=["fake", "real"]):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(image)
        prob = torch.sigmoid(output).item()
        pred = 1 if prob >= 0.5 else 0

    return class_names[pred], prob

# example
img_path = "/content/drive/MyDrive/XST-Deepfake-Detection/processed_ffpp/test/fake/sample.jpg"
label, prob = predict_image(model, img_path, val_test_transform, device)
print("Prediction:", label)
print("Probability:", prob)

## 7. Evaluate the Model

In [ ]:
test_loss, test_acc, test_precision, test_recall, test_f1, y_true, y_pred, y_prob = evaluate(
    model, test_loader, criterion, device
)

print("Test Results")
print(f"Loss      : {test_loss:.4f}")
print(f"Accuracy  : {test_acc:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"F1-score  : {test_f1:.4f}")

In [ ]:
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["fake", "real"], zero_division=0))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history["train_loss"], label="Train Loss")
plt.plot(history["val_loss"], label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history["train_acc"], label="Train Accuracy")
plt.plot(history["val_acc"], label="Val Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.show()

## 8. Save model to Google Drive

In [ ]:
save_path = "/content/drive/MyDrive/XST-Deepfake-Detection/best_cnn_vit_ffpp.pth"
torch.save(model.state_dict(), save_path)
print("Model saved to:", save_path)